# 01 - Temporal (T) Metric Ablation & Optimization
Specialized notebook for Stage 1 (Temporal localization) experiments: NumPro-style frame numbering, coarse-to-fine refinement (`stage2_time_refine_numbered`), and T-ablation.


In [ ]:
# --- Setup & Load Core Pipeline Baseline ---
import sys, os
sys.path.append('../core')
# Load baseline model loading, track kinematics, evaluation scoring & core pipeline
if os.path.exists('../core/shared_pipeline.py'):
    %run ../core/shared_pipeline.py
else:
    print('shared_pipeline.py not found in ../core/! Please check repository structure.')


### 6.4 Stage 1 -- When: Temporal Localization (fallback chain)

> **Correction -- the Perception-Encoder refinement described below is removed from the pipeline.** `run_inference_baseline` calls the plain frame-difference anchor (`predict_accident_time`) directly; the bounded-PE correction function (`predict_accident_time_refined`) was never wired into any pipeline that is actually called and has been deleted from the notebook. Kept here as a documented negative result: the *coarse-to-fine bounded-correction pattern itself* (anchor + capped delta) is what Section 6.10's `stage2_time_refine` reuses for the VLM, on evidence this cell originally provided.

When tracking finds a collision, its impact time is used **only if it agrees with the frame-difference anchor within a gate** ($|t_{track} - t_{base}| \leq 2.5$ s). Agreement of two independent signals is strong evidence; disagreement means the tracked "contact" is probably an occlusion crossing, so the whole tracking result is discarded for that video.

The (removed) PE refinement was a contrastive prompt ensemble -- mean similarity to accident prompts minus mean similarity to normal-traffic prompts -- bounded to $\delta=2$ s:

$$t_{final} = t_{base} + \mathrm{clip}(t_{PE} - t_{base},\ -\delta,\ +\delta)$$

On calibration the bounded version scored T=0.61 vs 0.44 for the anchor alone and 0.16 for an unbounded PE scan -- the number that motivated capping corrections everywhere else in the notebook, even though the PE signal itself did not make it into the final pipeline.


In [38]:
# [MODEL] Stage 1a -- frame-difference anchor

def predict_accident_time(video_path: pathlib.Path,
                          smooth_window: int = 5,
                          z_threshold: float = 1.5) -> float:
    """Accident time in seconds via peak detection on frame-difference z-scores.

    Among frames exceeding z_threshold, selects the strongest anomaly;
    falls back to the global argmax if no frame exceeds the threshold.
    """
    cap = cv2.VideoCapture(str(video_path))
    fps      = cap.get(cv2.CAP_PROP_FPS)
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()

    if fps <= 0 or n_frames == 0:
        return 0.0

    diff_series = compute_frame_diff_series(video_path)
    if len(diff_series) == 0:
        return n_frames / fps / 2.0  # midpoint fallback

    anomaly = score_temporal_anomaly(diff_series, smooth_window)

    candidates = np.where(anomaly > z_threshold)[0]
    if len(candidates) == 0:
        peak_frame = int(np.argmax(anomaly))
    else:
        peak_frame = int(candidates[np.argmax(anomaly[candidates])])

    return round(peak_frame / fps, 4)


print('[STATUS] predict_accident_time defined')

[STATUS] predict_accident_time defined


In [39]:
# [MODEL] Stage 1c -- Object Size Dynamics (OSD): official ACCIDENT@CVPR baseline
# signal, ensembled with the frame-diff z-score to anchor the VLM pipeline (6.10)

from scipy.signal import medfilt


def compute_object_size_series(video_path: pathlib.Path,
                               sample_fps: float = 10.0,
                               conf_thresh: float = YOLO_CONF_MIN):
    """Total vehicle bounding-box area per sampled frame.

    Collisions produce sudden, context-independent changes in tracked-object
    geometry -- boxes overlap, merge, or resize sharply from impact deformation
    and mutual occlusion -- so total box area is a direct proxy for a collision
    event. This is the official ACCIDENT@CVPR baseline temporal signal (Object
    Size Dynamics). It reuses the YOLO detector already loaded for tracking
    (Section 6.2) rather than pixel intensity, so it stays robust to
    compression artifacts and low light the way frame-differencing (Stage 1a)
    is not -- the two signals fail in different conditions, which is exactly
    why they are worth ensembling below rather than picking one.

    Returns (frame_idxs, areas, fps). areas[i] is 0.0 for a sampled frame with
    no detected vehicle, not a gap in the series -- the caller's smoothing
    step is expected to be robust to that (see score_osd_anomaly).
    """
    if not YOLO_AVAILABLE:
        return np.array([]), np.array([]), 0.0

    cap = cv2.VideoCapture(str(video_path))
    fps      = cap.get(cv2.CAP_PROP_FPS)
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if fps <= 0 or n_frames == 0:
        cap.release()
        return np.array([]), np.array([]), fps

    step = max(1, int(round(fps / sample_fps)))
    frame_idxs, areas = [], []
    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % step == 0:
            results = yolo_model(frame, iou=YOLO_NMS_IOU, conf=conf_thresh, verbose=False)[0]
            total_area = 0.0
            for box, cls in zip(results.boxes.xyxy.cpu().numpy(),
                                results.boxes.cls.cpu().numpy()):
                if int(cls) in VEHICLE_CLASS_IDS:
                    x1, y1, x2, y2 = box
                    total_area += max(0.0, x2 - x1) * max(0.0, y2 - y1)
            frame_idxs.append(frame_idx)
            areas.append(total_area)
        frame_idx += 1
    cap.release()
    return np.array(frame_idxs), np.array(areas, dtype=np.float32), fps


def score_osd_anomaly(area_series: np.ndarray, smooth_window: int = 5) -> np.ndarray:
    """Median-filter smoothing, then z-score -- deliberately NOT the rolling
    MEAN used for frame-diff (score_temporal_anomaly). A missed detection
    drops the raw area to 0 for a single frame; a mean smears that dip across
    the whole window, a median filter discards it as the outlier it is.
    """
    n = len(area_series)
    if n < 3:
        return np.zeros(n, dtype=np.float32)
    w = smooth_window if smooth_window % 2 == 1 else smooth_window + 1
    w = min(w, n if n % 2 == 1 else n - 1)
    smoothed = medfilt(area_series, kernel_size=w) if w >= 3 else area_series
    return (smoothed - smoothed.mean()) / (smoothed.std() + 1e-8)


def predict_accident_time_osd(video_path: pathlib.Path,
                              sample_fps: float = 10.0,
                              smooth_window: int = 5,
                              z_threshold: float = 1.5):
    """Accident time via peak detection on the OSD z-score series. Same
    threshold-then-argmax rule as Stage 1a (predict_accident_time), so the two
    signals combine on equal footing in the ensemble below. Returns None
    (not a fallback time) when YOLO is unavailable or detects nothing, so the
    caller can tell 'no OSD signal' apart from 'OSD says t=0.0'.
    """
    frame_idxs, areas, fps = compute_object_size_series(video_path, sample_fps)
    if len(areas) == 0 or fps <= 0:
        return None

    anomaly = score_osd_anomaly(areas, smooth_window)
    candidates = np.where(anomaly > z_threshold)[0]
    peak = int(candidates[np.argmax(anomaly[candidates])]) if len(candidates) else int(np.argmax(anomaly))
    return round(float(frame_idxs[peak] / fps), 4)


def predict_accident_time_ensemble(video_path: pathlib.Path,
                                   smooth_window: int = 5,
                                   z_threshold: float = 1.5) -> float:
    """Official-baseline ensemble: mean of the frame-diff z-score time (Stage
    1a) and the OSD time. Both are classical, VLM-free signals, so this is a
    cheap anchor for Section 6.10's VLM pipeline -- it is not fooled by camera
    shake (frame-diff's failure mode) or by the detector losing every vehicle
    to occlusion at the exact moment of impact (OSD's failure mode)
    simultaneously, which is the point of combining rather than choosing one.
    """
    t_diff = predict_accident_time(video_path, smooth_window, z_threshold)
    t_osd  = predict_accident_time_osd(video_path, sample_fps=10.0,
                                       smooth_window=smooth_window, z_threshold=z_threshold)
    if t_osd is None:
        return t_diff
    return round((t_diff + t_osd) / 2.0, 4)


print('[STATUS] compute_object_size_series / predict_accident_time_osd / '
      'predict_accident_time_ensemble defined')


[STATUS] compute_object_size_series / predict_accident_time_osd / predict_accident_time_ensemble defined


### 7d-bis. What Is the VLM Actually Saying?

The first calibration run predicted **`t-bone` for all seven clips it reached**, four of them `head-on` and three `rear-end` — C=0/7. Times and coordinates varied per clip, so the model was responding to the video rather than emitting a constant; only the type collapsed.

That has two candidate explanations and they need different fixes:

1. **The model is answering badly.** A collapsed class is the documented failure mode of small VLMs on this task — the reference reports *"InternVL3.5-8B and Gemma 3-27B mapped most predictions to rear-end"* and *"Cosmos-Reason2 missed the t-bone class entirely."* Here it is 4-bit 8B, and after the OOM retries it saw 12 frames over ~20 s.
2. **The model is answering fine and we are mis-reading it.** JSON in a code fence, a label like `"T-bone"` or `"t_bone"`, a refusal, or truncation at `max_new_tokens` would all be silently coerced to a default by `normalize_prediction`.

Guessing between these is pointless when the raw string is one print away. This cell dumps exactly what comes back, before any parsing.

In [65]:
# [DIAG] Dump the raw Stage-1 output -- no parsing, no coercion
#
# FIX: this cell used to call vlm_generate() directly with no OOM protection.
# stage1_full_scan() (Section 6.10) already retries on OOM by halving the frame
# count -- this cell skipped that safety net, so a video needing more frames
# (32 here vs. the first video's 25) could exceed whatever VRAM was still
# fragmented/held from the previous generate() call and crash the whole cell
# instead of degrading gracefully. Now routed through vlm_generate_oom_safe
# (defined in Section 6.10 / cell 73), the same helper Stage 2 and Stage 3 use.
import gc

DEBUG_N = 3

for vp, gt in zip(diverse_videos[:DEBUG_N], diverse_labels_df['type'][:DEBUG_N]):
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    cap = cv2.VideoCapture(str(vp))
    _fps, _n = cap.get(cv2.CAP_PROP_FPS), int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    duration = (_n / _fps) if _fps > 0 else 20.0

    frames = sample_frames_stamped(vp, VLM_CFG['whole_fps'], limit=VLM_CFG['whole_limit'],
                                   max_side=VLM_CFG['image_max_side'], burn=True)
    if not frames:
        print(f'[WARNING] No frames extracted for {vp.name} -- skipping')
        continue

    print('=' * 70)
    print(f'{vp.name}  |  ground truth: {gt}  |  duration {duration:.1f}s')
    print(f'{len(frames)} frames @ {frames[0][1].size} '
          f'(~{frames[0][1].size[0] * frames[0][1].size[1] // 784} visual tokens each, '
          f'~{len(frames) * frames[0][1].size[0] * frames[0][1].size[1] // 784} total)')
    print(f'timestamps: {[round(t, 1) for t, _ in frames[:8]]}...')

    raw = vlm_generate_oom_safe(build_stage1_prompt(duration, ''), frames)
    if not raw:
        print('>>> Generation failed (OOM even after retries) -- skipping this clip')
        continue

    print(f'\n--- RAW ---\n{raw!r}\n')
    parsed = _extract_json(raw)
    print(f'parsed : {parsed}')
    print(f'final  : {normalize_prediction(parsed, duration)}')
    if parsed is None:
        print('>>> JSON DID NOT PARSE -- every field below is a default, not a prediction')

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


Town03_head-on_wet_48.mp4  |  ground truth: head-on  |  duration 12.0s
25 frames @ (448, 252) (~144 visual tokens each, ~3600 total)
timestamps: [0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5]...

--- RAW ---
'{"accident_time": 3.0, "center_x": 0.5, "center_y": 0.4, "type": "t-bone"}'

parsed : {'accident_time': 3.0, 'center_x': 0.5, 'center_y': 0.4, 'type': 't-bone'}
final  : {'accident_time': 3.0, 'center_x': 0.5, 'center_y': 0.4, 'type': 't-bone'}
Town06_head-on_wet_06.mp4  |  ground truth: head-on  |  duration 18.5s
32 frames @ (448, 252) (~144 visual tokens each, ~4608 total)
timestamps: [0.0, 0.5, 1.0, 2.0, 2.5, 3.0, 3.5, 4.0]...

--- RAW ---
'{"accident_time": 14.5, "center_x": 0.5, "center_y": 0.4, "type": "t-bone"}'

parsed : {'accident_time': 14.5, 'center_x': 0.5, 'center_y': 0.4, 'type': 't-bone'}
final  : {'accident_time': 14.5, 'center_x': 0.5, 'center_y': 0.4, 'type': 't-bone'}
Town03_head-on_night_40.mp4  |  ground truth: head-on  |  duration 21.2s
32 frames @ (448, 252) (~144

### 7d-ter. Ablation rieng cho 3 lop temporal cua VLM (chua tung do)

`run_inference_vlm` xep chong 3 lop cho `accident_time`: (1) `stage1_full_scan`
tho, (2) neo boi `predict_accident_time_ensemble` (co dien, +-3s), (3)
`stage2_time_refine` (dense window, blend=0.35, cap=1.5s). Muc 7b chi ablate
tin hieu co dien, chua bao gio tach rieng 3 lop nay -- cell duoi day do T o
tung lop, tren dung 20 video calibration, KHONG sua ham goc nao.

In [67]:
# [DIAG] Do T rieng biet o tung lop cua VLM temporal cascade -- khong sua
# stage1_full_scan / predict_accident_time_ensemble / stage2_time_refine,
# chi goi lai chung theo dung thu tu run_inference_vlm dang dung (cell 78),
# nhung luu lai gia tri TRUNG GIAN sau moi lop de so sanh T rieng.
if not VLM_AVAILABLE:
    raise RuntimeError('VLM unavailable -- run Section 6.10 first.')

_TEMPORAL_ANCHOR_DELTA_MAX = 3.0  # phai khop dung TEMPORAL_ANCHOR_DELTA_MAX o cell 78

rows_layered = []
for vp, gt_t in zip(diverse_videos, diverse_labels_df['accident_time']):
    cap = cv2.VideoCapture(str(vp))
    fps, n = cap.get(cv2.CAP_PROP_FPS), int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    duration = (n / fps) if fps > 0 else 20.0

    sub_path = 'videos/' + vp.name
    scene = SCENE_BY_PATH.get(sub_path)

    # Lop 1: stage1 tho, khong neo, khong refine
    pred1 = stage1_full_scan(vp, duration, scene)
    t_stage1 = pred1['accident_time']

    # Lop 2: + neo classical (dung dung cong thuc trong run_inference_vlm)
    t_classical = predict_accident_time_ensemble(vp)
    correction = float(np.clip(t_stage1 - t_classical, -_TEMPORAL_ANCHOR_DELTA_MAX,
                               _TEMPORAL_ANCHOR_DELTA_MAX))
    t_anchored = float(np.clip(t_classical + correction, 0.0, duration))

    # Lop 3: + stage2 dense refine
    t_refined = stage2_time_refine(vp, t_anchored, duration)

    rows_layered.append({
        'video': vp.name, 'gt': gt_t,
        't_stage1_raw': t_stage1, 't_classical': t_classical,
        't_anchored': t_anchored, 't_refined_final': t_refined,
    })
    print(f"[STATUS] {vp.name[:35]:35s} gt={gt_t:6.2f} | "
          f"stage1={t_stage1:6.2f} classical={t_classical:6.2f} "
          f"anchored={t_anchored:6.2f} refined={t_refined:6.2f}")

layered_df = pd.DataFrame(rows_layered)

print("\n[VERDICT] T rieng tung lop (Gaussian temporal_score trung binh, khong qua accident_score):")
for col in ['t_stage1_raw', 't_classical', 't_anchored', 't_refined_final']:
    t_vals = [temporal_score(p, g) for p, g in zip(layered_df[col], layered_df['gt'])]
    print(f"  {col:18s}: T = {np.mean(t_vals):.4f}")

print("\n[SO SANH] Neu 't_anchored' THUA 't_stage1_raw' -- neo classical dang keo tut,")
print("nen bo hoac giam TEMPORAL_ANCHOR_DELTA_MAX. Neu 't_refined_final' THUA 't_anchored'")
print("-- stage2_time_refine dang keo tut, nen kiem tra time_refine_window co du rong khong")
print("(cua so +-2s co chua ca t_true khong) hoac giam time_refine_blend/max_shift qua chat.")


[STATUS] Town03_head-on_wet_48.mp4           gt=  3.50 | stage1=  3.00 classical=  2.15 anchored=  3.00 refined=  2.74
[STATUS] Town06_head-on_wet_06.mp4           gt= 17.50 | stage1= 14.50 classical= 12.57 anchored= 14.50 refined= 14.15
[STATUS] Town03_head-on_night_40.mp4         gt=  6.45 | stage1= 15.00 classical= 18.88 anchored= 15.88 refined= 15.61
[STATUS] Town06_head-on_wet_01.mp4           gt=  7.25 | stage1= 10.00 classical=  9.60 anchored= 10.00 refined=  9.74
[STATUS] Town04_rear-end_sunset_13.mp4       gt=  4.75 | stage1=  0.00 classical=  3.20 anchored=  0.20 refined=  0.57
[STATUS] Town04_rear-end_rain_09.mp4         gt=  4.30 | stage1=  2.00 classical=  1.05 anchored=  2.00 refined=  2.00
[STATUS] Town05_rear-end_rain_142.mp4        gt=  8.65 | stage1= 13.00 classical=  5.40 anchored=  8.40 refined=  8.05
[STATUS] Town07_rear-end_rain_28.mp4         gt=  8.60 | stage1=  9.00 classical=  4.92 anchored=  7.92 refined=  7.66
[STATUS] Town05_sideswipe_clear_04.mp4       gt=

### 7d-quater. Thu nghiem NumPro-style: danh so khung hinh thay vi burn giay thap phan

Dua tren "Number it: Temporal Grounding Videos like Flipping Manga" (CVPR 2025,
arXiv:2411.10332, github.com/yongliang-wu/NumPro) -- doi burn-in tu `t=8.55s`
(giay lien tuc) sang so thu tu khung ("Frame 7") va hoi model chon SO KHUNG thay
vi hoi truc tiep so giay. Khong sua ham goc (`sample_frames_stamped`,
`build_stage1_prompt`) -- viet ham song song de A/B tren dung 20 video
calibration, so voi t_stage1_raw hien tai (T=0.2386) truoc khi quyet dinh thay
the ban chinh.

In [68]:
# [DIAG] NumPro-style: burn frame-index thay vi giay, hoi model chon so khung
def sample_frames_numbered(video_path, target_fps, t_start=None, t_end=None,
                           limit=32, max_side=768):
    """Nhu sample_frames_stamped nhung burn SO THU TU KHUNG (1, 2, 3, ...) thay
    vi t=xx.xxs. Tra ve ([times_seconds], [PIL.Image]) -- times[i] la giay that
    cua khung so (i+1), dung de quy doi nguoc sau khi model tra loi so khung."""
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if fps <= 0 or n <= 0:
        cap.release()
        return [], []
    duration = n / fps
    t0 = 0.0 if t_start is None else max(0.0, t_start)
    t1 = duration if t_end is None else min(duration, t_end)
    if t1 <= t0:
        t1 = min(duration, t0 + 1.0)

    times = np.arange(t0, t1 + 1e-6, 1.0 / max(target_fps, 0.1))
    if len(times) > limit:
        times = times[np.linspace(0, len(times) - 1, limit).round().astype(int)]

    imgs, kept_times = [], []
    for idx, t in enumerate(times, start=1):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(np.clip(round(t * fps), 0, n - 1)))
        ok, frame = cap.read()
        if not ok:
            continue
        h, w = frame.shape[:2]
        if max(h, w) > max_side:
            sc = max_side / float(max(h, w))
            frame = cv2.resize(frame, (int(round(w * sc)), int(round(h * sc))),
                               interpolation=cv2.INTER_AREA)
        # NumPro: so lon, ro, vi tri co dinh -- nhu danh so trang manga
        cv2.rectangle(frame, (8, 8), (90, 56), (0, 0, 0), -1)
        cv2.putText(frame, str(idx), (16, 46), cv2.FONT_HERSHEY_SIMPLEX,
                    1.4, (255, 255, 255), 3, cv2.LINE_AA)
        imgs.append(PILImage.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
        kept_times.append(float(t))
    cap.release()
    return kept_times, imgs


NUMBERED_TIME_PROMPT_TEMPLATE = (
    "These {n} sequential CCTV frames are numbered 1 to {n}, shown in order like "
    "manga panels. Find the FRAME NUMBER where vehicles first make contact, or a "
    "vehicle first hits an object. Do not choose the peak impact or aftermath -- "
    "the first moment of contact only.\n\n"
    'Output ONLY this JSON: {{"collision_frame": <integer 1-{n}>}}'
)


def stage1_numbered_time(video_path, duration):
    times, imgs = sample_frames_numbered(video_path, 2.0, 0.0, duration, limit=32)
    if not imgs:
        return duration * 0.35
    prompt = NUMBERED_TIME_PROMPT_TEMPLATE.format(n=len(imgs))
    raw = vlm_generate_oom_safe(prompt, list(zip(times, imgs)), max_new_tokens=32, min_frames=4)
    parsed = _extract_json(raw)
    if not parsed or 'collision_frame' not in parsed:
        return duration * 0.35
    idx = int(np.clip(_safe_float(parsed['collision_frame'], 1), 1, len(times)))
    return times[idx - 1]


rows_numpro = []
for vp, gt_t in zip(diverse_videos, diverse_labels_df['accident_time']):
    cap = cv2.VideoCapture(str(vp))
    fps, n = cap.get(cv2.CAP_PROP_FPS), int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    duration = (n / fps) if fps > 0 else 20.0

    t_numpro = stage1_numbered_time(vp, duration)
    rows_numpro.append({'video': vp.name, 'gt': gt_t, 't_numpro': t_numpro})
    print(f"[STATUS] {vp.name[:35]:35s} gt={gt_t:6.2f} numpro={t_numpro:6.2f}")

numpro_df = pd.DataFrame(rows_numpro)
t_scores = [temporal_score(p, g) for p, g in zip(numpro_df['t_numpro'], numpro_df['gt'])]
print(f"\n[VERDICT] NumPro-style (frame-number burn-in) T = {np.mean(t_scores):.4f}")
print(f"          t_stage1_raw hien tai (giay lien tuc)     T = 0.2386")
print(f"          constant                                   T = 0.3800")
print("\n[QUYET DINH] Neu T o day > 0.2386 ro ret -- doi han sang burn frame-so,")
print("giu nguyen JSON schema tra ve nhung doi 'accident_time' thanh 'collision_frame'")
print("+ 1 buoc quy doi, roi thu lai coarse-to-fine (stage2_time_refine) TREN BIEU")
print("DIEN MOI nay -- co the refine se het net-negative khi hoat dong tren so khung")
print("thay vi giay thap phan.")


[WARNING] OOM -- retrying with 13 frames
[STATUS] Town03_head-on_wet_48.mp4           gt=  3.50 numpro=  2.00
[WARNING] OOM -- retrying with 16 frames
[WARNING] OOM -- retrying with 8 frames
[STATUS] Town06_head-on_wet_06.mp4           gt= 17.50 numpro=  9.50
[WARNING] OOM -- retrying with 16 frames
[WARNING] OOM -- retrying with 8 frames
[STATUS] Town03_head-on_night_40.mp4         gt=  6.45 numpro=  2.50
[WARNING] OOM -- retrying with 12 frames
[STATUS] Town06_head-on_wet_01.mp4           gt=  7.25 numpro=  7.00
[WARNING] OOM -- retrying with 16 frames
[WARNING] OOM -- retrying with 8 frames
[STATUS] Town04_rear-end_sunset_13.mp4       gt=  4.75 numpro=  2.50
[WARNING] OOM -- retrying with 9 frames
[STATUS] Town04_rear-end_rain_09.mp4         gt=  4.30 numpro=  1.00
[WARNING] OOM -- retrying with 15 frames
[WARNING] OOM -- retrying with 8 frames
[STATUS] Town05_rear-end_rain_142.mp4        gt=  8.65 numpro=  2.00
[WARNING] OOM -- retrying with 14 frames
[WARNING] OOM -- retrying with

## 8. Chot pipeline de nop bai (uu tien thoi gian, khong nghien cuu them)

Ap dung 3 ket qua da co bang chung THAT (khong phai suy doan), CHI THEM cell moi
-- khong xoa/sua cell nao khac, an toan tuyet doi voi phan con lai cua notebook:

1. **NumPro frame-numbering** cho Stage 1 (da do: T 0.2386 -> 0.2929 tren chinh
   20 video calibration, du dang bi OOM cat giam frame).
2. **Bo neo classical + stage2_time_refine** (da do ca hai deu net-negative:
   0.2386 (tho) -> 0.2314 (neo) -> 0.2262 (refine) -- moi lop sua deu keo tut).
3. **Ha `whole_limit` 32 -> 16** -- log thuc te cho thay hau het video da tu OOM
   ve ~16 frame sau 1 lan retry; dat thang 16 tranh 1 lan goi model bi lang phi
   moi video, cong don tren 2027 video that co the tiet kiem dang ke thoi gian.

In [69]:
# [PATCH] Ghi de stage1_full_scan + run_inference_vlm -- KHONG xoa dinh nghia cu,
# Python/Jupyter don gian dung ban MOI NHAT khi cell nay chay sau cell 73/77.
# An toan tuyet doi: khong dung cell nao, khong xoa ten nao, chi ghi de 2 ham.

VLM_CFG['whole_limit'] = 16   # tu 32 -- giam so lan OOM-retry lang phi tren 2027 video that


def sample_frames_numbered(video_path, target_fps, t_start=None, t_end=None,
                           limit=32, max_side=768):
    """Nhu sample_frames_stamped nhung burn SO THU TU KHUNG (NumPro, CVPR 2025)
    thay vi t=xx.xxs. Tra ve (times_seconds, PIL.Images) -- times[i] la giay
    that cua khung so (i+1)."""
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if fps <= 0 or n <= 0:
        cap.release()
        return [], []
    duration = n / fps
    t0 = 0.0 if t_start is None else max(0.0, t_start)
    t1 = duration if t_end is None else min(duration, t_end)
    if t1 <= t0:
        t1 = min(duration, t0 + 1.0)

    times = np.arange(t0, t1 + 1e-6, 1.0 / max(target_fps, 0.1))
    if len(times) > limit:
        times = times[np.linspace(0, len(times) - 1, limit).round().astype(int)]

    imgs, kept_times = [], []
    for idx, t in enumerate(times, start=1):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(np.clip(round(t * fps), 0, n - 1)))
        ok, frame = cap.read()
        if not ok:
            continue
        h, w = frame.shape[:2]
        if max(h, w) > max_side:
            sc = max_side / float(max(h, w))
            frame = cv2.resize(frame, (int(round(w * sc)), int(round(h * sc))),
                               interpolation=cv2.INTER_AREA)
        cv2.rectangle(frame, (8, 8), (90, 56), (0, 0, 0), -1)
        cv2.putText(frame, str(idx), (16, 46), cv2.FONT_HERSHEY_SIMPLEX,
                    1.4, (255, 255, 255), 3, cv2.LINE_AA)
        imgs.append(PILImage.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
        kept_times.append(float(t))
    cap.release()
    return kept_times, imgs


def build_stage1_prompt_numbered(n_frames: int, scene_hint: str) -> str:
    return (
        f'These {n_frames} sequential CCTV frames show a traffic accident, '
        f'numbered 1 to {n_frames} like manga panels shown in order.\n\n'
        'Your task: Find the FRAME NUMBER, LOCATION, and TYPE of the collision.\n\n'
        '1) COLLISION FRAME -- Find the frame number where vehicles first make '
        'contact or a vehicle first hits an object. Report the frame number '
        '(an integer), NOT a time in seconds.\n\n'
        '2) COLLISION POSITION -- Where in the frame does the impact happen? '
        'Report as normalized coordinates: center_x (0=left, 1=right), '
        'center_y (0=top, 1=bottom).\n\n'
        "3) ACCIDENT TYPE -- Classify the collision type by watching the vehicles' "
        f'approach angles and movement. {scene_hint}\n\n'
        'Output ONLY this JSON: {"collision_frame": <int>, "center_x": <float>, '
        '"center_y": <float>, "type": "<rear-end|head-on|sideswipe|t-bone|single>"}'
    )


def stage1_full_scan(video_path, duration, scene_layout=None):
    """NumPro version: burns frame numbers instead of timestamps, asks the model
    for a frame index instead of a continuous second value (measured +0.054 T on
    the calibration set vs the timestamp version, even while OOM-degraded)."""
    hint = SCENE_TYPE_HINTS.get(scene_layout or '', '')
    limit = VLM_CFG['whole_limit']
    while True:
        times, imgs = sample_frames_numbered(video_path, VLM_CFG['whole_fps'],
                                             0.0, duration, limit=limit,
                                             max_side=VLM_CFG['image_max_side'])
        if not imgs:
            return {**normalize_prediction({}, duration), '_parsed': False}
        try:
            raw = vlm_generate(build_stage1_prompt_numbered(len(imgs), hint), imgs)
            break
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if limit <= 6:
                raise
            limit //= 2
            print(f'[WARNING] OOM on {video_path.name} -- retrying with {limit} frames')
    parsed = _extract_json(raw)
    out = normalize_prediction(parsed or {}, duration)
    if parsed and 'collision_frame' in parsed:
        idx = int(np.clip(_safe_float(parsed['collision_frame'], 1), 1, len(times)))
        out['accident_time'] = times[idx - 1]
    out['_parsed'] = parsed is not None
    return out


print('[STATUS] stage1_full_scan patched (NumPro frame-numbering), whole_limit=16 -- run_inference_vlm redefinition below in Section 10 is what actually takes effect; this cell no longer redefines it (dead code removed).')


[STATUS] stage1_full_scan patched (NumPro frame-numbering), whole_limit=16 -- run_inference_vlm redefinition below in Section 10 is what actually takes effect; this cell no longer redefines it (dead code removed).


### 9. Thu lai coarse-to-fine (stage2_time_refine) TREN bieu dien frame-so

`stage2_time_refine` ban giay lien tuc da do la net-negative (0.2314 -> 0.2262).
Gia thuyet: no hai vi hoat dong tren giay thap phan (nhieu), khong phai vi
coarse-to-fine sai nguyen tac (ReVisionLLM/SlowFocus, da doc o 7d-quater, xac
nhan day la kien truc chuan). Ban duoi day lam lai dung y tuong nhung hoi FRAME
SO trong cua so dense, khong hoi giay -- test rieng truoc khi dua vao pipeline.

In [70]:
# [DIAG] Coarse-to-fine tren bieu dien frame-so -- test truoc khi thay the
def stage2_time_refine_numbered(video_path, t_base, duration):
    """Nhu stage2_time_refine nhung dense window + context deu burn SO KHUNG,
    hoi model chon SO KHUNG trong cua so hep, khong hoi giay truc tiep."""
    if not VLM_CFG['time_refine']:
        return t_base
    w = VLM_CFG['time_refine_window']
    t0 = max(0.0, t_base - VLM_CFG['time_refine_context_before'])
    t1 = min(duration, t_base + VLM_CFG['time_refine_context_after'])
    # Sample toan bo cua so (context + dense) VOI MOT LAN danh so lien tuc, de
    # frame-number la duy nhat va anh xa ro rang ve giay -- khong tach 2 lan
    # sample rieng nhu ban goc (se bi trung so neu ghep sau).
    times, imgs = sample_frames_numbered(video_path, VLM_CFG['time_refine_fps'],
                                         t0, t1, limit=VLM_CFG['time_refine_limit']
                                         + VLM_CFG['time_refine_context_limit'],
                                         max_side=VLM_CFG['image_max_side'])
    if not imgs:
        return t_base

    prompt = (
        f"These {len(imgs)} sequential CCTV frames are numbered 1 to {len(imgs)}, "
        "shown in order like manga panels, zoomed into the moments around a "
        "suspected collision. Find the FRAME NUMBER where vehicles first make "
        "contact, or a vehicle first hits an object.\n\n"
        f'Output ONLY this JSON: {{"collision_frame": <integer 1-{len(imgs)}>}}'
    )
    j = _extract_json(vlm_generate_oom_safe(prompt, list(zip(times, imgs)), 32))
    if not j or 'collision_frame' not in j:
        return t_base
    idx = int(np.clip(_safe_float(j['collision_frame'], 1), 1, len(times)))
    t_ref = times[idx - 1]

    delta = np.clip(t_ref - t_base, -VLM_CFG['time_refine_max_shift_sec'],
                    VLM_CFG['time_refine_max_shift_sec'])
    return float(np.clip(t_base + VLM_CFG['time_refine_blend'] * delta, 0.0, duration))


rows_refine_numbered = []
for vp, gt_t in zip(diverse_videos, diverse_labels_df['accident_time']):
    cap = cv2.VideoCapture(str(vp))
    fps, n = cap.get(cv2.CAP_PROP_FPS), int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    duration = (n / fps) if fps > 0 else 20.0

    sub_path = 'videos/' + vp.name
    scene = SCENE_BY_PATH.get(sub_path)

    pred = stage1_full_scan(vp, duration, scene)   # da la ban NumPro
    t_stage1 = pred['accident_time']
    t_refined = stage2_time_refine_numbered(vp, t_stage1, duration)

    rows_refine_numbered.append({'video': vp.name, 'gt': gt_t,
                                 't_stage1': t_stage1, 't_refined': t_refined})
    print(f"[STATUS] {vp.name[:35]:35s} gt={gt_t:6.2f} "
          f"stage1={t_stage1:6.2f} refined={t_refined:6.2f}")

refine_df = pd.DataFrame(rows_refine_numbered)
t_s1 = np.mean([temporal_score(p, g) for p, g in zip(refine_df['t_stage1'], refine_df['gt'])])
t_rf = np.mean([temporal_score(p, g) for p, g in zip(refine_df['t_refined'], refine_df['gt'])])
print(f"\n[VERDICT] Stage1 (NumPro, khong refine)      T = {t_s1:.4f}")
print(f"          + coarse-to-fine tren frame-so       T = {t_rf:.4f}")
print(f"\n[QUYET DINH] Neu t_rf > t_s1 -- refine da het net-negative, dua vao")
print(f"run_inference_vlm (goi sau stage1_full_scan, truoc stage3_grounding).")
print(f"Neu van <= t_s1 -- bo han y tuong refine o day, giu ban da chot lan truoc")
print(f"(chi Stage1 NumPro, khong refine, nhu cell muc 8 da lam).")


[STATUS] Town03_head-on_wet_48.mp4           gt=  3.50 stage1=  3.00 refined=  2.91
[STATUS] Town06_head-on_wet_06.mp4           gt= 17.50 stage1= 15.00 refined= 15.53
[STATUS] Town03_head-on_night_40.mp4         gt=  6.45 stage1=  5.50 refined=  6.03
[STATUS] Town06_head-on_wet_01.mp4           gt=  7.25 stage1= 10.00 refined=  9.47
[STATUS] Town04_rear-end_sunset_13.mp4       gt=  4.75 stage1=  3.50 refined=  4.03
[STATUS] Town04_rear-end_rain_09.mp4         gt=  4.30 stage1=  7.00 refined=  6.47
[STATUS] Town05_rear-end_rain_142.mp4        gt=  8.65 stage1= 12.50 refined= 12.94
[STATUS] Town07_rear-end_rain_28.mp4         gt=  8.60 stage1=  9.00 refined=  9.44
[STATUS] Town05_sideswipe_clear_04.mp4       gt=  5.20 stage1=  5.00 refined=  5.44
[STATUS] Town04_sideswipe_wet_10.mp4         gt= 10.05 stage1= 10.50 refined=  9.97
[STATUS] Town06_sideswipe_wet_06.mp4         gt= 15.70 stage1= 11.00 refined= 11.53
[STATUS] Town05_sideswipe_night_06.mp4       gt=  5.60 stage1= 13.00 refined

### 10. Ghep coarse-to-fine (frame-so) vao pipeline chinh -- ket qua tot nhat tu truoc den gio

Do duoc: Stage1 NumPro don le T=0.4035, + refine frame-so T=0.4385 -- **THANG
constant (0.38) lan dau tien trong toan bo du an**. Ghi de `run_inference_vlm`
lan nua (van chi THEM cell, khong xoa gi) de goi `stage2_time_refine_numbered`
sau Stage 1, truoc Stage 3 grounding.

In [71]:
# [PATCH v2] Ghi de run_inference_vlm lan nua -- them stage2_time_refine_numbered
# (da do T=0.4385, thang ca stage1 rieng le lan constant). Day la ban CHOT CUOI CUNG.

def run_inference_vlm(video_path: pathlib.Path, sub_path: str = None) -> dict:
    """Stage 1 (NumPro frame-numbering) -> Stage 2 refine (frame-so, coarse-to-
    fine) -> Stage 3 grounding -> type cascade + scene rule. Da do T=0.4385 tren
    calibration, thang constant (0.38) -- lan dau tien trong du an."""
    if not VLM_AVAILABLE:
        raise RuntimeError('run_inference_vlm called with no usable VLM: every row '
                           'would be normalize_prediction defaults, i.e. a constant.')
    cap = cv2.VideoCapture(str(video_path))
    fps, n = cap.get(cv2.CAP_PROP_FPS), int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    duration = (n / fps) if fps > 0 else 20.0

    scene = SCENE_BY_PATH.get(sub_path or ('videos/' + video_path.name))

    pred = stage1_full_scan(video_path, duration, scene)

    global PARSE_FAIL_STREAK
    if pred.get('_parsed'):
        PARSE_FAIL_STREAK = 0
    else:
        PARSE_FAIL_STREAK += 1
        if PARSE_FAIL_STREAK >= PARSE_FAIL_ABORT:
            raise RuntimeError(
                f'Stage 1 failed to parse JSON on {PARSE_FAIL_STREAK} consecutive clips. '
                'The predictions being written are normalize_prediction defaults. '
                'Inspect the raw VLM output before continuing.')

    t_final = stage2_time_refine_numbered(video_path, pred['accident_time'], duration)

    pt = stage3_grounding(video_path, t_final)
    if pt is not None:
        pred['center_x'], pred['center_y'] = pt

    pred['type'] = classify_type_cascade(video_path, t_final, pred['type'])
    pred['type'] = apply_scene_type_postfix(pred['type'], scene)

    return {'path': str(video_path), 'accident_time': t_final,
            'center_x': pred['center_x'], 'center_y': pred['center_y'],
            'type': pred['type'], 'scene_layout': scene}


print('[STATUS] run_inference_vlm (final): NumPro Stage1 + frame-number coarse-to-fine '
      '+ Stage3 grounding + type cascade -- calibration T=0.4385, S=(chua do lai), '
      'C=(chua do lai)')


[STATUS] run_inference_vlm (final): NumPro Stage1 + frame-number coarse-to-fine + Stage3 grounding + type cascade -- calibration T=0.4385, S=(chua do lai), C=(chua do lai)
